In [1]:
import pandas as pd
import string, json, datetime

def to_float(value):
    """
    Converts a value to float. 
    Returns 0.0 if the value is None, empty, or not a valid number.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None
    
    try:
        clean_val = str(value).replace(',', '').strip()
        return float(clean_val)
    except (ValueError, TypeError):
        return None

def to_int(value):
    """
    Converts a value to int. 
    Returns 0 if the value is None, empty, or not a valid number.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None
    
    try:
        clean_val = str(value).replace(',', '').strip()
        return int(clean_val)
    except (ValueError, TypeError):
        return None
    
def to_bool(value):
    """
    Converts a value to boolean.
    Returns None if the value is None or empty.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None
    
    str_val = str(value).strip().lower()
    if str_val in ['true', '1', 'yes']:
        return True
    elif str_val in ['false', '0', 'no']:
        return False
    else:
        return None

def process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns):
    """
    Flattens all columns in the dataframe into a simple key:value JSON structure.
    Uses filename as main key instead of workorder_id.
    """
    df = df.rename(columns=rename_columns)
    exclude_from_data = ["workorder_id", "filename"] + exclude_cols

    for _, row in df.iterrows():
        fname = str(row.get("filename", "unknown"))
        wo_id = row.get("workorder_id", None)

        if not fname or fname == "nan":
            continue

        if fname not in final_json:
            final_json[fname] = {
                "workorder_id": wo_id,
                "data": {}
            }

        row_flattened_data = {}

        for col in df.columns:
            if col in exclude_from_data:
                continue

            value = row[col]

            if any(num_col in col for num_col in float_columns):
                row_flattened_data[col] = to_float(value)
            elif isinstance(value, (pd.Timestamp, datetime.datetime, datetime.date)):
                row_flattened_data[col] = value.isoformat()
            elif any(num_col in col for num_col in int_columns):
                row_flattened_data[col] = to_int(value)
            elif any(bool_col in col for bool_col in bool_columns):
                row_flattened_data[col] = to_bool(value)
            else:
                row_flattened_data[col] = value if pd.notna(value) else None

        final_json[fname]["data"].update(row_flattened_data)

    return final_json

### UPS System

In [4]:
# 'checklist_items.servicing_and_cleaning.j.status': 'servicing_and_cleaning.j.status',
# 'checklist_items.servicing_and_cleaning.m.status': 'servicing_and_cleaning.m.status',
# 'checklist_items.servicing_and_cleaning.n.status': 'servicing_and_cleaning.n.status',

In [5]:
new_columns = {
    'safety_and_preparation.make_sure_safety_precautions_procedures_being_followed_as_required.status': 'safety_and_preparation.safety_precautions.status',
    'safety_and_preparation.inform_occ_before_start_any_task.status': 'safety_and_preparation.inform_occ.status',
    
    'general_inspection.perform_visual_inspection_and_confirm_all_cable_properly_connected.status': 'general_inspection.visual_inspection.status',
    'general_inspection.confirm_both_of_module_is_on_and_inverter_mode.status': 'general_inspection.confirm_both_of_module_and_inverter.status',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.status': 'general_inspection.record_reading.status',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.A_battery_capacity': 'general_inspection.readings.A_battery_capacity',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.B_module_temp.in': 'general_inspection.readings.B_module_temperature.min',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.B_module_temp.out': 'general_inspection.readings.B_module_temperature.max',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.C_battery_run_time': 'general_inspection.readings.C_battery_run_time',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.D_output_freq_hz': 'general_inspection.readings.D_output_frequency',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.E_bypass_freq_hz': 'general_inspection.readings.E_bypass_frequency',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.F_battery_voltage_v.positive': 'general_inspection.readings.F_battery_voltage_v.positive',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.F_battery_voltage_v.negative': 'general_inspection.readings.F_battery_voltage_v.negative',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.G_battery_charge_car_a.positive': 'general_inspection.readings.G_battery_charge_car.positive',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.G_battery_charge_car_a.negative': 'general_inspection.readings.G_battery_charge_car.negative',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.H_discharge_current_a': 'general_inspection.readings.H_discharge_current',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.I_rectifier_voltage_v_a': 'general_inspection.readings.I_rectifier_voltage.a',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.I_rectifier_voltage_v_b': 'general_inspection.readings.I_rectifier_voltage.b',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.I_rectifier_voltage_v_c': 'general_inspection.readings.I_rectifier_voltage.c',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.J_bypass_voltage_v_a': 'general_inspection.readings.J_bypass_voltage.a',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.J_bypass_voltage_v_b': 'general_inspection.readings.J_bypass_voltage.b',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.J_bypass_voltage_v_c': 'general_inspection.readings.J_bypass_voltage.c',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.K_output_voltage_v_a': 'general_inspection.readings.K_output_voltage.a',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.K_output_voltage_v_b': 'general_inspection.readings.K_output_voltage.b',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.K_output_voltage_v_c': 'general_inspection.readings.K_output_voltage.c',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.L_output_current_a_a': 'general_inspection.readings.L_output_current.a',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.L_output_current_a_b': 'general_inspection.readings.L_output_current.b',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.L_output_current_a_c': 'general_inspection.readings.L_output_current.c',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.M_active_power_kw_a': 'general_inspection.readings.M_active_power.a',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.M_active_power_kw_b': 'general_inspection.readings.M_active_power.b',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.M_active_power_kw_c': 'general_inspection.readings.M_active_power.c',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.N_reactive_power_kva_a': 'general_inspection.readings.N_reactive_power.a',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.N_reactive_power_kva_b': 'general_inspection.readings.N_reactive_power.b',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.N_reactive_power_kva_c': 'general_inspection.readings.N_reactive_power.c',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.O_apparent_power_kva_a': 'general_inspection.readings.O_apparent_power.a',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.O_apparent_power_kva_b': 'general_inspection.readings.O_apparent_power.b',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.O_apparent_power_kva_c': 'general_inspection.readings.O_apparent_power.c',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.P_output_power_percent_a': 'general_inspection.readings.P_output_power.a',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.P_output_power_percent_b': 'general_inspection.readings.P_output_power.b',
    'general_inspection.using_scroll_arrow_scroll_and_record_as_below.readings.P_output_power_percent_c': 'general_inspection.readings.P_output_power.c',
    
    'servicing_and_cleaning.set_the_ups_to_maintenance_bypass_mode_by_using_command_from_module.status': 'servicing_and_cleaning.set_maintenance_bypass_mode.status',
    'servicing_and_cleaning.switching_on_switch_maintenance_bypass_in_front_of_ups_rack.status': 'servicing_and_cleaning.switch_on_maintenance_bypass.status',
    'servicing_and_cleaning.confirm_the_display_panel_is_showing_bypass_mode.status': 'servicing_and_cleaning.confirm_display_panel.status',
    'servicing_and_cleaning.switch_off_all_battery_breakers.status': 'servicing_and_cleaning.switch_off_all_battery_breakers.status',
    'servicing_and_cleaning.switch_off_both_of_module_using_2_finger_push_button_on_off.status': 'servicing_and_cleaning.switch_off_both_of_module.status',
    'servicing_and_cleaning.open_screw_and_pull_both_of_module_to_cleaning.status': 'servicing_and_cleaning.open_screw_and_pull_both_module.status',
    'servicing_and_cleaning.open_screws_each_battery_bank_and_check_the_voltage.status': 'servicing_and_cleaning.open_battery_bank_and_check_voltage.status',
    'servicing_and_cleaning.wait_for_the_electrolytic_condensers_on_the_power_board_to_discharge_wait_at_least_15min_before_proceed_any_maintenance_work.status': 'servicing_and_cleaning.wait_electrolytic_condensers_discharge.status',
    'servicing_and_cleaning.h.battery_readings.battery1': 'servicing_and_cleaning.wait_electrolytic_condensers_discharge.battery_readings.battery1',
    'servicing_and_cleaning.h.battery_readings.battery2': 'servicing_and_cleaning.wait_electrolytic_condensers_discharge.battery_readings.battery2',
    'servicing_and_cleaning.h.battery_readings.battery3': 'servicing_and_cleaning.wait_electrolytic_condensers_discharge.battery_readings.battery3',
    'servicing_and_cleaning.randomly_check_the_voltages_on_the_batteries._ensure_minimum_voltage_is_read_as_12v_and_maximum_should_not_be_more_than_13.8v.status': 'servicing_and_cleaning.randomly_check_the_voltages_on_the_batteries.status',
    'servicing_and_cleaning.check_end_to_end_batteries_voltages.status': 'servicing_and_cleaning.check_end_to_end_batteries_voltages.status',
    'servicing_and_cleaning.do_the_cleaning_on_all_the_batteries._confirm_and_ensure_no_cable_is_disconnected_from_any_terminals.status': 'servicing_and_cleaning.do_the_cleaning_on_all_the_batteries.status',
    'servicing_and_cleaning.after_completed_the_cleaning_confirm_all_cablings_are_secured_and_in_the_correct_position.status': 'servicing_and_cleaning.confirm_all_cables_after_cleaning.status',
    
    'servicing_and_cleaning.switch_on_both_of_module_using_2_finger_push_button_on_off.status': 'servicing_and_cleaning.switch_on_both_module.status',
    'servicing_and_cleaning.button_on_off.status': 'servicing_and_cleaning.button_on_off.status',
    'servicing_and_cleaning.normalized_all_battery_breakers.status': 'servicing_and_cleaning.normalized_all_battery_breakers.status',
    'servicing_and_cleaning.switch_off_switch_maintenance_bypass_in_front_of_ups_rack.status': 'servicing_and_cleaning.switch_off_switch_maintenance_bypass.status',
    'servicing_and_cleaning.set_the_ups_to_inverter_mode_by_using_command_from_module.status': 'servicing_and_cleaning.set_the_ups_to_inverter_mode.status',
    'servicing_and_cleaning.confirm_with_occ_no_alarm_is_activated_on_scada.status': 'servicing_and_cleaning.confirm_with_occ_no_alarm_is_activated_on_scada.status',
    'servicing_and_cleaning.check_and_see_is_everything_back_to_normal.status': 'servicing_and_cleaning.check_everything__normal.status',
    
    'battery_backup_test.before_proceed_this_test_make_sure_battery_level_should_be_above_80%_of_load.status': 'battery_backup_test.check_battery_level.status',
    'battery_backup_test.confirm_ups_in_inverter_mode_and_no_alarm_activated.status': 'battery_backup_test.confirm_ups_in_inverter_mode_and_no_alarm_activated.status',
    'battery_backup_test.turn_off_main_switch_breaker_for_main_incoming_supply.status': 'battery_backup_test.turn_off_main_switch_breaker.status',
    'battery_backup_test.confirm_after_a_few_seconds_alarm_sounding_"battery_working"_led_illuminate_and_ups_change_to_"battery_mode".status': 'battery_backup_test.confirm_alarm_sounding.status',
    'battery_backup_test.observe_the_ups_time_remaining_and_battery_percentage.status': 'battery_backup_test.observe_the_ups_time_remaining_and_battery_percentage.status',
    'battery_backup_test.switch_on_back_main_switch_breaker_after_ups_reaching_50%_of_battery_level.status': 'battery_backup_test.switch_on_back_main_switch_breaker.status',
    'battery_backup_test.battery_backup_test.before.time_minutes': 'battery_backup_test.before.time_minutes',
    'battery_backup_test.battery_backup_test.before.battery_percentage': 'battery_backup_test.before.battery',
    'battery_backup_test.battery_backup_test.after.time_minutes': 'battery_backup_test.after.time_minutes',
    'battery_backup_test.battery_backup_test.after.battery_percentage': 'battery_backup_test.after.battery',
    'battery_backup_test.battery_backup_test.battery_backup_time_minutes': 'battery_backup_test.battery_backup_time',

    'performed_by.name': 'technician_name',
    'performed_by.id': 'technician_id',
    'verified_by.name': 'supervisor_name',
    'verified_by.id': 'supervisor_id',
 }

In [6]:
path = "../../output/psd/ups_system.xlsx" 
df = pd.read_excel(path, sheet_name="ups_system", keep_default_na=False)

bool_columns = []
float_columns = []
int_columns = []
final_json = {}
exclude_cols = []
rename_columns = {}

rename_dict = {}

for col in df.columns:
    new_col = col

    if new_col.startswith("checklist_items."):
        new_col = new_col.replace("checklist_items.", "")
    if new_col.endswith("..status"):
        new_col = new_col.replace("..status", ".status")
    if new_col != col:
        rename_dict[col] = new_col

df = df.rename(columns=rename_dict)
df = df.rename(columns=new_columns)
df.drop(columns=['servicing_and_cleaning.j.status', 'servicing_and_cleaning.m.status', 'servicing_and_cleaning.n.status', 'pm_order', 'reference_document'], inplace=True)

df.columns

bool_columns = [
    'safety_and_preparation.safety_precautions.status',
    'safety_and_preparation.inform_occ.status',
    'general_inspection.visual_inspection.status',
    'general_inspection.confirm_both_of_module_and_inverter.status',
    'general_inspection.record_reading.status',
    'servicing_and_cleaning.set_maintenance_bypass_mode.status',
    'servicing_and_cleaning.switch_on_maintenance_bypass.status',
    'servicing_and_cleaning.confirm_display_panel.status',
    'servicing_and_cleaning.switch_off_all_battery_breakers.status',
    'servicing_and_cleaning.switch_off_both_of_module.status',
    'servicing_and_cleaning.open_screw_and_pull_both_module.status',
    'servicing_and_cleaning.open_battery_bank_and_check_voltage.status',
    'servicing_and_cleaning.wait_electrolytic_condensers_discharge.status',
    'servicing_and_cleaning.check_end_to_end_batteries_voltages.status',
    'servicing_and_cleaning.do_the_cleaning_on_all_the_batteries.status',
    'servicing_and_cleaning.confirm_all_cables_after_cleaning.status',
    'servicing_and_cleaning.switch_on_both_module.status',
    'servicing_and_cleaning.button_on_off.status',
    'servicing_and_cleaning.normalized_all_battery_breakers.status',
    'servicing_and_cleaning.switch_off_switch_maintenance_bypass.status',
    'servicing_and_cleaning.set_the_ups_to_inverter_mode.status',
    'servicing_and_cleaning.confirm_with_occ_no_alarm_is_activated_on_scada.status',
    'servicing_and_cleaning.check_everything__normal.status',
    'battery_backup_test.check_battery_level.status',
    'battery_backup_test.confirm_ups_in_inverter_mode_and_no_alarm_activated.status',
    'battery_backup_test.turn_off_main_switch_breaker.status',
    'battery_backup_test.confirm_alarm_sounding.status',
    'battery_backup_test.observe_the_ups_time_remaining_and_battery_percentage.status',
    'battery_backup_test.switch_on_back_main_switch_breaker.status',
]

float_columns = [
    'general_inspection.readings.A_battery_capacity',
    'general_inspection.readings.B_module_temperature.min',
    'general_inspection.readings.B_module_temperature.max',
    'general_inspection.readings.C_battery_run_time',
    'general_inspection.readings.D_output_frequency',
    'general_inspection.readings.E_bypass_frequency',
    'general_inspection.readings.F_battery_voltage_v.positive',
    'general_inspection.readings.F_battery_voltage_v.negative',
    'general_inspection.readings.G_battery_charge_car.positive',
    'general_inspection.readings.G_battery_charge_car.negative',
    'general_inspection.readings.H_discharge_current',
    'general_inspection.readings.I_rectifier_voltage.a',
    'general_inspection.readings.I_rectifier_voltage.b',
    'general_inspection.readings.I_rectifier_voltage.c',
    'general_inspection.readings.J_bypass_voltage.a',
    'general_inspection.readings.J_bypass_voltage.b',
    'general_inspection.readings.J_bypass_voltage.c',
    'general_inspection.readings.K_output_voltage.a',
    'general_inspection.readings.K_output_voltage.b',
    'general_inspection.readings.K_output_voltage.c',
    'general_inspection.readings.L_output_current.a',
    'general_inspection.readings.L_output_current.b',
    'general_inspection.readings.L_output_current.c',
    'general_inspection.readings.M_active_power.a',
    'general_inspection.readings.M_active_power.b',
    'general_inspection.readings.M_active_power.c',
    'general_inspection.readings.N_reactive_power.a',
    'general_inspection.readings.N_reactive_power.b',
    'general_inspection.readings.N_reactive_power.c',
    'general_inspection.readings.O_apparent_power.a',
    'general_inspection.readings.O_apparent_power.b',
    'general_inspection.readings.O_apparent_power.c',
    'general_inspection.readings.P_output_power.a',
    'general_inspection.readings.P_output_power.b',
    'general_inspection.readings.P_output_power.c',
    'servicing_and_cleaning.wait_electrolytic_condensers_discharge.battery_readings.battery1',
    'servicing_and_cleaning.wait_electrolytic_condensers_discharge.battery_readings.battery2',
    'servicing_and_cleaning.wait_electrolytic_condensers_discharge.battery_readings.battery3',
    'battery_backup_test.before.time_minutes',
    'battery_backup_test.before.battery',
    'battery_backup_test.after.time_minutes',
    'battery_backup_test.after.battery',
    'battery_backup_test.battery_backup_time',
]

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for filename, payload in final_json.items():
    rows.append({
        "filename": filename,
        "workorder_no": payload.get("workorder_id"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/psd/response_ups_system.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

Saved as: ../../output/psd/response_ups_system.xlsx


### Station Inspection

In [6]:
import re, pandas as pd, json

# genset_path = "../../output/psd/genset_inspection.xlsx"
genset_path = "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Coding/python.notebook.extraction/technician_pm/extracted/genset_inspection_finale (1) (1).xlsx"
station_path = "../../output/psd/station_and_switch.xlsx"

df_genset = pd.read_excel(genset_path, sheet_name="genset_inspection", keep_default_na=False)
df_genset.columns = ['genset.' + col for col in df_genset.columns]
df_genset.drop_duplicates(subset=['genset.workorder_id'], inplace=True)

df_station = pd.read_excel(station_path, sheet_name="station_and_switch", keep_default_na=False)
df_station.columns = ['station.' + col for col in df_station.columns]
df_station.drop_duplicates(subset=['station.workorder_id'], inplace=True)

df = pd.merge(df_genset, df_station, left_on="genset.workorder_id", right_on="station.workorder_id", how="outer", suffixes=('_genset', '_station'))
df.duplicated(subset=['genset.workorder_id']).value_counts()

df.drop(columns=[
    'genset.filename', 'station.inspection_date', 'station.workorder_id', 'station.station_location', 
    'station.perform_by.name', 'station.perform_by.id', 'station.perform_by.technician', 'station.perform_by.date', 
    'station.verify_by.name', 'station.verify_by.id', 'station.verify_by.technician', 'station.verify_by.date', 
    'station.reference_document', 'station.start_time', 'station.end_time'
    ], inplace=True)

df['station.filename'] = df.apply(
    lambda row: f"PS_PM_WEK_StationInspection_{row['genset.workorder_id']}.pdf" if pd.isna(row['station.filename']) or row['station.filename'] == '' else row['station.filename'],
    axis=1
)

cols = df.columns.tolist()

front_cols = ['genset.station_location', 'genset.inspection_date']
station_cols = [c for c in cols if c.startswith("station.")]
genset_cols = [c for c in cols if c.startswith("genset.") and c not in front_cols]
other_cols   = [c for c in cols if not (c.startswith("station.") or c.startswith("genset.")) and c not in front_cols]

df = df[front_cols + station_cols + genset_cols + other_cols]

final_json = {}

exclude_cols = ["genset.perform_by.technician", "genset.verify_by.technician"]

rename_columns = {
    "genset.workorder_id": "workorder_id",
    "station.filename": "filename",
    "genset.station_location": "station_location",
    "genset.inspection_date": "inspection_date",
    "genset.perform_by.name": "technician_name",
    "genset.perform_by.id": "technician_id",
    "genset.perform_by.date": "technician_date",
    "genset.verify_by.name": "supervisor_name",
    "genset.verify_by.id": "supervisor_id",
    "genset.verify_by.date": "supervisor_date",
}

float_columns = [
    "genset.battery_and_battery_charger.battery_voltage",
    "station.main_switch_board.voltmeter",
    "station.main_switch_board.ammeter", "station.main_switch_board.cos",
    "station.main_switch_board.capacitor_bank", "station.essential_main.voltmeter",
]

int_columns = [
    "genset.diesel_engine.running_hour.before", 
    "genset.diesel_engine.running_hour.after",
    "genset.amf_board.output_service.phase_to_phase_reading.ry",
    "genset.amf_board.output_service.phase_to_phase_reading.yb",
    "genset.amf_board.output_service.phase_to_phase_reading.rb",
    "genset.amf_board.output_service.phase_to_neutral.rn",
    "genset.amf_board.output_service.phase_to_neutral.yn"
]

bool_columns = []

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

# for filename, payload in final_json.items():
#     rows.append({
#         "filename": filename,
#         "workorder_no": payload.get("workorder_id"),
#         "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
#     })

for filename, payload in final_json.items():
    workorder_no = payload.get("workorder_id")
    
    if pd.isna(workorder_no) or not str(workorder_no).strip() or str(workorder_no).strip().lower() == 'nan':
        match = re.search(r'\d+', str(filename))
        if match:
            workorder_no = match.group(0) # Extracts '4000679188'
            
    rows.append({
        "filename": filename,
        "workorder_no": int(workorder_no),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })


df_out = pd.DataFrame(rows)

# output_file = f"../../output/psd/response_station_inspection.xlsx"
# df_out.to_excel(output_file, index=False)

output_file = f"../../output/psd/response_station_inspection_new.csv"
df_out.to_csv(output_file, index=False)

# print(f"Saved as: {output_file}")


### TPSS Inspection

In [ ]:
path = "../../output/psd/tpss_inspection_weekly.xlsx" 

df = pd.read_excel(path, sheet_name="tpss_inspection_weekly", keep_default_na=False)
final_json = {}

exclude_cols = ["perform_by.technician", "verify_by.engineer/supervisor", "reference_document"]

rename_columns = {
    "perform_by.name": "technician_name",
    "perform_by.id": "technician_id",
    "perform_by.date": "technician_date",
    "verify_by.name": "supervisor_name",
    "verify_by.id": "supervisor_id",
    "verify_by.date": "supervisor_date",
}

float_columns = [
    "11kv_switchgear.incomer1.voltmeter", "11kv_switchgear.incomer1.ammeter",
    "11kv_switchgear.incomer2.voltmeter", "11kv_switchgear.incomer2.ammeter",
    "11kv_switchgear.rectifier_transformer.voltmeter", "11kv_switchgear.rectifier_transformer.ammeter",
    "11kv_switchgear.service_transformer.voltmeter", "11kv_switchgear.service_transformer.ammeter",
    "750_dc_switchgear.rectifier_s/gear.voltage", "750_dc_switchgear.rectifier_s/gear.ampere",
    "750_dc_switchgear.feeder1.ampere", "750_dc_switchgear.feeder2.ampere", "750_dc_switchgear.feeder3.ampere", "750_dc_switchgear.feeder4.ampere", "750_dc_switchgear.aaru.ampere",
    "battery_&_battery_charger.voltage", "battery_&_battery_charger.ampere",
    "main_switch_board.voltmeter_reading.value",
    "main_switch_board.ammeter_reading.value",
    "main_switch_board.cos_reading.value"
]

int_columns = []
bool_columns = []

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for filename, payload in final_json.items():
    rows.append({
        "filename": filename,
        "workorder_no": payload.get("workorder_id"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/psd/response_tpss_inspection_weekly.xlsx"
df_out.to_csv(output_file, index=False)

print(f"Saved as: {output_file}")


Saved as: ../../output/psd/response_tpss_inspection_weekly.csv


### Blue Light System

In [9]:
import string
from collections import defaultdict

def rename_inspection_columns(df):
    counter = defaultdict(int)     # counts per main group
    pair_map = {}                  # (group, description) -> letter
    new_columns = []

    for col in df.columns:
        # 1. remove inspection.
        col_clean = col.replace("inspection.", "", 1)

        parts = col_clean.split(".")

        # expect: group.description.status|remarks
        if len(parts) < 3:
            new_columns.append(col_clean)
            continue

        group = parts[0]           # deformation_discoloration
        description = parts[1]     # long sentence
        suffix = parts[-1]         # status / remarks

        key = (group, description)

        # assign SAME letter for status + remarks
        if key not in pair_map:
            letter = string.ascii_lowercase[counter[group]]
            pair_map[key] = letter
            counter[group] += 1
        else:
            letter = pair_map[key]

        new_columns.append(f"{group}.{letter}.{suffix}")

    df = df.copy()
    df.columns = new_columns
    return df


In [10]:
path = "../../output/psd/blue_light_system.xlsx" 

df = pd.read_excel(path, sheet_name="blue_light_system", keep_default_na=False)
final_json = {}

exclude_cols = []

rename_columns = {
    "date_time": "inspection_date",
    "perform_by.technician_id": "technician_id",
    "verified_by.supervisor_id": "supervisor_id",
}

float_columns = []
int_columns = []
bool_columns = [
    "inspection.external_condition.check_whether_there_is_any_damage_on_the_panel_surface._if_any,_repair_the_damaged_part.status",
    "inspection.sound_vibration.check_whether_there_is_abnormal_sound_or_vibration_from_the_panel._if_any,_identify_the_abnormal_part_and_repair_or_replace_it.status",
    "inspection.odor.check_whether_there_is_abnormal_smell_from_the_panel._if_any,_identify_the_abnormal_parts_and_repair_or_replace_it.status",
    "inspection.dust.if_there_is_much_dust_outside_the_panel,_clean_it.status",
    "inspection.rust.if_metal_component_rust_away,_check_humidity_and_dew_condensation.status",
    "inspection.rust.replace_the_rusted_component_if_necessary.status",
    "inspection.painting.check_whether_painting_comes_out.status",
    "inspection.painting.touch_up_the_places_where_painting_came_off_badly.status",
    "inspection.loosen_connection.check_the_screw_condition_of_connection_point._if_any_fallen_looseness_bolt,_tighten_the_bolt.status",
    "inspection.deformation_discoloration.check_deformation_and_discoloration_of_components.status",
    "inspection.deformation_discoloration.if_any,_replace_the_component.status",
    "inspection.loose_of_connection.check_the_screw_condition_of_connection_point._if_any_fallen_looseness_bolt,_tighten_the_bolt.status"
]

df = rename_inspection_columns(df)
process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for filename, payload in final_json.items():
    rows.append({
        "filename": filename,
        "workorder_no": payload.get("workorder_id"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/psd/response_blue_light_system.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")


Saved as: ../../output/psd/response_blue_light_system.xlsx
